# 100. Same Tree
**Difficulty:** 🟢 Easy · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/same-tree/

## 💡 Concepts

**Core concept(s):** Walk **both trees together** with DFS (or an explicit stack).

**Why it applies here:** Two trees are identical only if their roots match and their left subtrees match and their right subtrees match — the same question asked in lockstep on both.

**Key intuition:** Compare the two roots, then compare left-with-left and right-with-right the same way.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- Recursion on a pair of nodes.

## 📝 Problem

Return `True` if two trees have the same shape **and** the same values.

**Example**
```
[1,2,3] vs [1,2,3] -> True
[1,2]   vs [1,None,2] -> False   (different shape)
```

> Two approaches, both `O(n)`: recursion vs an explicit stack.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Recursion

**Idea:** Both empty → same. One empty or values differ → different. Otherwise recurse on both children.

**Time:** `O(n)`. **Space:** `O(h)`.

In [ ]:
def same_rec(p: Optional[TreeNode], q: Optional[TreeNode]) -> bool:
    if not p and not q:
        return True                        # both empty here -> matching so far
    if not p or not q or p.val != q.val:
        return False                       # one empty, or values differ -> not the same
    # Values match; now the left subtrees must match AND the right subtrees must match.
    return same_rec(p.left, q.left) and same_rec(p.right, q.right)

### Approach 2 — Explicit Stack

**Idea:** Same logic, but keep pairs of nodes on a stack instead of using recursion.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
def same_iter(p: Optional[TreeNode], q: Optional[TreeNode]) -> bool:
    stack = [(p, q)]                       # pairs of nodes still to compare
    while stack:
        a, b = stack.pop()                 # take the next pair
        if not a and not b:
            continue                       # both empty here -> fine, check other pairs
        if not a or not b or a.val != b.val:
            return False                   # mismatch found
        stack.append((a.left, b.left))     # queue the left children to compare
        stack.append((a.right, b.right))   # queue the right children to compare
    return True

In [ ]:
# Correctness check
tests = [([1,2,3],[1,2,3],True), ([1,2],[1,None,2],False), ([],[],True), ([1,2,1],[1,1,2],False)]
for a, b, exp in tests:
    r1, r2 = build_tree(a), build_tree(b)
    x, y = same_rec(r1, r2), same_iter(r1, r2)
    print(f"{a} vs {b} -> rec={x}, iter={y} | expected={exp}")
    assert x == y == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    t = build_balanced(n)
    return (t, build_balanced(n))   # identical trees -> full comparison
solutions = {
    "recursion O(n)": same_rec,
    "stack     O(n)": same_iter,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Parallel traversal:** walk two structures in lockstep, comparing matching positions.
- **Signal:** "are these two trees equal", "mirror of", "structurally identical".
- **Related problems:** Symmetric Tree, Subtree of Another Tree, Merge Two Binary Trees.
- **Common pitfalls:** (1) checking values before the both-empty / one-empty cases; (2) comparing left-with-right by accident.